# DeepGuard — WildDeepfake full download

One-click download of the full WildDeepfake test data from the public Hugging Face mirror. The pilot has already confirmed that this route works. Files are stored directly on Google Drive.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, json, hashlib, time
drive.mount('/content/drive', force_remount=False)
ROOT=Path('/content/drive/MyDrive/DeepGuard')
BASE=ROOT/'datasets/WildDeepfake/full'
ARCH=BASE/'archives'
OUT=BASE/'extracted'
ARCH.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
print('Destination:',BASE)
print('Free Drive GiB:',round(shutil.disk_usage('/content/drive').free/1024**3,1))
assert shutil.disk_usage('/content/drive').free > 15*1024**3, 'Less than 15 GB free on Drive'


In [ ]:
!pip -q install -U huggingface_hub
from huggingface_hub import list_repo_files, hf_hub_download
repo='xingjunm/WildDeepfake'
all_files=list_repo_files(repo_id=repo,repo_type='dataset')
targets=[f for f in all_files if f.startswith('deepfake_in_the_wild/') and f.endswith('.tar.gz')]
print('Archives found:',len(targets))
print('First:',targets[:5])
assert len(targets)>100, 'Unexpectedly small repository listing; stop rather than download incomplete data.'
(BASE/'remote_files.json').write_text(json.dumps(targets,indent=2))


In [ ]:
# Download every archive directly to Drive. Existing files are skipped.
manifest_path=BASE/'download_manifest.json'
manifest=json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
for i,remote in enumerate(targets,1):
    name=remote.replace('deepfake_in_the_wild/','').replace('/','__')
    target=ARCH/name
    if target.exists() and target.stat().st_size>0:
        print(f'[{i}/{len(targets)}] existing {name}')
        continue
    print(f'[{i}/{len(targets)}] downloading {remote}')
    cached=hf_hub_download(repo_id=repo,filename=remote,repo_type='dataset',local_dir=str(ARCH),local_dir_use_symlinks=False)
    p=Path(cached)
    if p != target: shutil.move(str(p),str(target))
    manifest[remote]={'local':str(target),'bytes':target.stat().st_size,'sha256':hashlib.sha256(target.read_bytes()).hexdigest()}
    manifest_path.write_text(json.dumps(manifest,indent=2))
print('Download phase complete. Archives on Drive:',len(list(ARCH.glob('*.tar.gz'))))


In [ ]:
# Extract to Drive, one archive at a time; skip non-empty existing destinations.
import tarfile
for i,p in enumerate(sorted(ARCH.glob('*.tar.gz')),1):
    dest=OUT/p.name[:-7]
    marker=dest/'.EXTRACTED_OK'
    if marker.exists():
        print(f'[{i}] already extracted {p.name}')
        continue
    dest.mkdir(parents=True,exist_ok=True)
    print(f'[{i}] extracting {p.name}')
    with tarfile.open(p,'r:gz') as tf: tf.extractall(dest)
    marker.write_text('ok')
print('Extraction complete.')


In [ ]:
# Final inventory and fake/real counts.
from collections import Counter
counts=Counter(); total=0
for p in OUT.rglob('*'):
    if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png'}:
        total += p.stat().st_size
        s=str(p).lower()
        counts['fake' if 'fake_test' in s else 'real' if 'real_test' in s else 'other'] += 1
print('Images:',sum(counts.values()))
print('Counts:',dict(counts))
print('Extracted GiB:',round(total/1024**3,2))
print('DONE — full WildDeepfake is ready for DeepGuard testing.')
